## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-19-05-52-32 +0000,nyt,Here’s the latest.,https://www.nytimes.com/live/2026/11/18/world/...
1,2026-11-19-05-51-37 +0000,nyt,Uncontacted Awá of Brazil Emerge From Rainfore...,https://www.nytimes.com/2026/11/18/world/ameri...
2,2026-11-19-05-48-05 +0000,nyt,Undoing Brexit,https://www.nytimes.com/2026/11/18/world/19int...
3,2026-11-19-05-42-00 +0000,wsj,Asian Equities Mixed Amid Cautious Market Sent...,https://www.wsj.com/finance/asian-equities-dec...
4,2026-11-19-05-23-57 +0000,wapo,Conflicting agendas and a weakened Trump mark ...,https://www.washingtonpost.com/politics/2026/1...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2751/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
21,trump,64
255,new,24
104,china,22
39,nvidia,20
36,xi,20


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
272,2026-11-18-18-41-56 +0000,chitri,President Trump meets Xi Jinping and Vladimir ...,https://www.chicagotribune.com/2026/11/18/trum...,166
27,2026-11-19-03-53-50 +0000,chitri,President Trump meets Xi and Putin in China wi...,https://www.chicagotribune.com/2026/11/18/trum...,163
5,2026-11-19-05-20-04 +0000,startribune,Nuclear arms and ending wars in Iran and Ukrai...,https://www.startribune.com/trump-meets-xi-and...,163
192,2026-11-18-21-39-00 +0000,wsj,A top House Democrat issued document-preservat...,https://www.wsj.com/politics/policy/democrats-...,154
322,2026-11-18-16-10-44 +0000,bbc,Trump shakes hands with Putin and Xi ahead of ...,https://www.bbc.co.uk/news/articles/ck9d75x5ww...,147


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
272,166,2026-11-18-18-41-56 +0000,chitri,President Trump meets Xi Jinping and Vladimir ...,https://www.chicagotribune.com/2026/11/18/trum...
56,98,2026-11-19-02-00-00 +0000,wsj,Jensen Huang’s ouster from Nvidia is giving ne...,https://www.wsj.com/politics/national-security...
192,80,2026-11-18-21-39-00 +0000,wsj,A top House Democrat issued document-preservat...,https://www.wsj.com/politics/policy/democrats-...
78,79,2026-11-19-01-11-00 +0000,wsj,Elon Musk said he felt “pretty foolish” about ...,https://www.wsj.com/politics/policy/elon-musk-...
166,62,2026-11-18-22-13-12 +0000,cbc,She says Russell Brand sexually assaulted her ...,https://www.cbc.ca/radio/asithappens/mia-latte...
51,62,2026-11-19-02-20-15 +0000,bbc,AI giant Anthropic valued at up to $1.5tn ahea...,https://www.bbc.co.uk/news/articles/c6742m39xg...
130,53,2026-11-18-23-11-17 +0000,nypost,Iran and US trade tit-for-tat strikes as wild ...,https://nypost.com/2026/11/18/world-news/uae-o...
228,50,2026-11-18-20-38-09 +0000,nypost,Tech giant worth $170B axes hundreds of Califo...,https://nypost.com/2026/11/18/tech/chipmaker-q...
108,50,2026-11-19-00-09-28 +0000,chitri,"Illinois Supreme Court picks Jennifer Soble, I...",https://www.chicagotribune.com/2026/11/18/jenn...
415,48,2026-11-18-10-23-57 +0000,bbc,Democrats win Alaska Senate race two weeks aft...,https://www.bbc.co.uk/news/articles/cqev3v2ygz...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
